# Data Preparation - Olist Marketplace
Daten strukturiert und reproduzierbar verarbeiten

## Verbindung mit Duckdb

In [2]:
import duckdb
from pathlib import Path
import pandas as pd

# Verbindung zur DuckDB (in-memory reicht völlig)
con = duckdb.connect()

# Hilfsfunktion für SQL-Abfragen
def sql(q):
    return con.sql(q).df()

# Projektpfad bestimmen (Notebook liegt in /notebooks)
DATA = Path("../data/raw/brazilian-ecommerce")

In [3]:
# Alle CSV-Dateien in DuckDB registrieren
for f in DATA.glob("*.csv"):
    name = f.stem.replace("olist_", "").replace("_dataset", "")
    
    con.sql(f"""
        CREATE OR REPLACE TABLE {name} AS
        SELECT * FROM read_csv_auto('{f.as_posix()}')
        """
           )
           
# Alle Tabellen anzeigen
sql("SHOW TABLES")

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category_name_translation
7,products
8,sellers


## Erstellung der Dataframes pro Kernaufgabe

### EDA für erste Kernaufgabe

In [8]:
# Dataframe für Aufgabe 1. für EDA
df_rfm_eda = sql("""
SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    oi.price,
    oi.freight_value,
    Count(oi.product_id) AS product_count,
    
FROM customers AS c
JOIN orders o 
        ON c.customer_id = o.customer_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
GROUP BY c.customer_id, o.order_id, o.order_purchase_timestamp, 
                 o.order_approved_at, oi.price, oi.freight_value, 
                 c.customer_unique_id, c.customer_city, c.customer_state, 
                 c.customer_zip_code_prefix, o.order_status
    """)

In [9]:
df_rfm_eda

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
0,3a874b4d4c4b6543206ff5d89287f0c3,a25d5f94840d3c6a1a49f271ed83f4ec,rio de janeiro,RJ,21715,fbf9ac61453ac646ce8ad9783d7d0af6,delivered,2018-02-20 23:46:53,2018-02-22 02:30:46,109.90,15.53,1
1,6454e6cba392b35aa21527063026fc92,3ed766ba830792bcc9ad889d2607eab7,sao bernardo do campo,SP,09811,3923e3ade70348985bd2ca389905cf19,delivered,2018-03-07 23:00:33,2018-03-09 03:00:35,59.90,18.25,1
2,0fc25a3451d81cfe409466e229afdbb0,43088e9ae1468ae7f8f6bd222b6a7329,taubate,SP,12080,cbec453ddd874ca620b8afc6393e3218,delivered,2017-06-04 10:05:59,2017-06-04 10:22:32,89.90,13.65,1
3,e26eb46e8146dedecc6524e5d9771047,26ad164ad913649d86d05c20c3eaad57,osasco,SP,06086,74c951ce0835b2741d3014321ae9c480,delivered,2018-08-20 17:38:55,2018-08-20 17:50:28,129.90,9.66,1
4,eec909ab0da04f0d3e92f898c6994d37,00dd4390a8e8ad7a126e8331decc49ce,rio de janeiro,RJ,20540,b693d6c35867fa1937455e04f2547df2,delivered,2017-05-11 09:13:17,2017-05-11 09:25:21,94.90,14.41,1
...,...,...,...,...,...,...,...,...,...,...,...,...
101565,de1c64a7f4179d4216ee09d181aa2e6a,dde8dd36a5290a1bfb650e1c8037b88b,brasilia,DF,70340,d181dfce5d8355bc51c2370c6910d4bc,delivered,2017-07-06 17:25:17,2017-07-07 17:30:19,238.81,19.63,1
101566,7429462259e9d39e60127253ab7bdbe7,5bb8cdfb477b265ee4a460fcdfa5d0e4,praia grande,SP,11719,3a650c50958d54a9beebc3db169500cd,delivered,2018-02-13 23:19:30,2018-02-13 23:30:32,29.45,9.34,1
101567,e5426fc9fcee3dfa52fb048f6d0856e9,770c46cc8437a67264254f23c1786b97,campinas,SP,13084,6dce05516fc9a1e16b7a70e49ab656c2,delivered,2017-05-14 11:05:08,2017-05-14 11:15:13,26.14,9.36,1
101568,fbfb33cc34116ccf95b043f7cd31c692,1d7c9872acb163f50b958fa4b729dd02,santo antonio de padua,RJ,28470,4294d6c2b09c5f2438c01b11c91ff7a8,delivered,2018-06-10 13:42:58,2018-06-10 13:55:13,24.00,37.06,1


In [10]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
count,101570,101556,101570.000000,101570.000000,101570.000000
mean,2018-01-01 00:42:04.180673,2018-01-01 12:06:22.988666,124.922151,20.140526,1.109087
min,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.000000
25%,2017-09-13 08:05:38,2017-09-13 17:27:48.500000,40.800000,13.160000,1.000000
50%,2018-01-19 16:57:45,2018-01-20 09:09:20.500000,79.000000,16.340000,1.000000
75%,2018-05-04 23:26:44,2018-05-05 12:55:31.500000,139.530000,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.680000,20.000000
std,NaN,NaN,189.479405,15.901388,0.474137


#### Auffälligkeiten

1. Bei order_purchase_timestamp und order_approved_at scheint es NaN werte zu geben
2. 

In [ ]:
df_rfm_eda.dtypes

customer_id                         object
customer_unique_id                  object
customer_city                       object
customer_state                      object
customer_zip_code_prefix            object
order_id                            object
order_status                        object
order_purchase_timestamp    datetime64[us]
order_approved_at           datetime64[us]
payment_value                      float64
payment_type                        object
payment_installments                 int64
payment_sequential                   int64
price                              float64
freight_value                      float64
product_id                          object
dtype: object

In [11]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = {'customer_id': 'category', 
              'customer_unique_id': 'category', 
              'customer_city': 'category', 
              'customer_state': 'category', 
              'customer_zip_code_prefix': 'category', 
              'order_id': 'category', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'price': 'float32',
              'freight_value': 'float32',
              'product_count': 'int16', }
df_rfm_eda = df_rfm_eda.astype(col_dtypes)
df_rfm_eda.dtypes

customer_id                      category
customer_unique_id               category
customer_city                    category
customer_state                   category
customer_zip_code_prefix         category
order_id                         category
order_status                     category
order_purchase_timestamp    datetime64[s]
order_approved_at           datetime64[s]
price                             float32
freight_value                     float32
product_count                       int16
dtype: object

In [12]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
count,101570,101556,101570.000000,101570.000000,101570.000000
mean,2018-01-01 00:42:04,2018-01-01 12:06:22,124.922150,20.140526,1.109087
min,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.000000
25%,2017-09-13 08:05:38,2017-09-13 17:27:48,40.799999,13.160000,1.000000
50%,2018-01-19 16:57:45,2018-01-20 09:09:20,79.000000,16.340000,1.000000
75%,2018-05-04 23:26:44,2018-05-05 12:55:31,139.529995,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.679993,20.000000
std,NaN,NaN,189.479401,15.901388,0.474137


In [13]:
df_rfm_eda.isna().sum()

customer_id                  0
customer_unique_id           0
customer_city                0
customer_state               0
customer_zip_code_prefix     0
order_id                     0
order_status                 0
order_purchase_timestamp     0
order_approved_at           14
price                        0
freight_value                0
product_count                0
dtype: int64

In [14]:
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
7039,0bf35cac6cc7327065da879e2d90fae8,c4c0011e639bdbcf26059ddc38bd3c18,varzea paulista,SP,13225,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,NaT,28.990000,10.960000,1
7570,d85919cb3c0529589c6fa617f5f43281,c094ac95fcd52f821809ec232a7a6956,sao vendelino,RS,95795,3c0b8706b065f9919d0505d3b3343881,delivered,2017-02-17 15:53:27,NaT,133.990005,23.200001,1
11780,68d081753ad4fe22fc4d410a9eb1ca01,2e0a2166aa23da2472c6a60c4af6f7a6,sao paulo,SP,03573,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaT,149.800003,13.630000,1
29679,07a2a7e0f63fd8cb757ed77d4245623c,79af1bbf230a2630487975aa5d7d6220,paraisopolis,MG,37660,51eb2eebd5d76a24625b31c33dd41449,delivered,2017-02-18 15:52:27,NaT,59.900002,17.160000,1
35327,29c35fc91fc13fb5073c8f30505d860d,7e1a5ca61b572d76b64b6688b9f96473,caninde,CE,62700,5cf925b116421afa85ee25e99b4c34fb,delivered,2017-02-18 16:48:35,NaT,79.989998,26.820000,1
37330,d5de688c321096d15508faae67a27051,d49f3dae6bad25d05160fc17aca5942d,conselheiro lafaiete,MG,36400,7002a78c79c519ac54022d4f8a65e6e8,delivered,2017-01-19 22:26:59,NaT,45.900002,14.520000,1
39826,2127dc6603ac33544953ef05ec155771,8a9a08c7ca8900a200d83cf838a07e0b,cotia,SP,06708,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,309.899994,39.110001,1
40155,684cb238dc5b5d6366244e0e0776b450,6ff8b0d7b35d5c945633b8d60165691b,santos,SP,11030,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,39.990002,14.520000,1
40787,f67cd1a215aae2a1074638bbd35a223a,bc1896dc77f49e6dec880445a9b443a3,rio de janeiro,RJ,21020,88083e8f64d95b932164187484d90212,delivered,2017-02-18 22:49:19,NaT,49.000000,14.520000,2
51968,74bebaf46603f9340e3b50c6b086f992,f79be7c08dd24b72d34634f1b89333a4,sao jose de ribamar,MA,65110,2babbb4b15e6d2dfe95e2de765c97bce,delivered,2017-02-18 17:15:03,NaT,79.989998,26.820000,1


In [15]:
pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_approved_at'].isna())

order_approved_at,False,True
order_status,,
approved,2,0
canceled,464,0
delivered,99341,14
invoiced,319,0
processing,304,0
shipped,1119,0
unavailable,7,0


In [16]:
df_rfm_eda['order_approved_at'] = df_rfm_eda['order_approved_at'].fillna(
    pd.to_datetime(df_rfm_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count


In [17]:
print("Gesamte Duplikate:", df_rfm_eda.duplicated().sum())

Gesamte Duplikate: 0


In [18]:
test = pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_id']).T
test


order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable
order_id,,,,,,,
00010242fe8c5a6d1ba2dd792cb16214,0,0,1,0,0,0,0
00018f77f2f0320c557190d7a144bdd3,0,0,1,0,0,0,0
000229ec398224ef6ca0657da4fc703e,0,0,1,0,0,0,0
00024acbcdf0a6daa1e931b038114c75,0,0,1,0,0,0,0
00042b26cf59d7ce69dfabb4e55b4fd9,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...
fffc94f6ce00a00581880bf54a75a037,0,0,1,0,0,0,0
fffcd46ef2263f404302a634eb57f7eb,0,0,1,0,0,0,0
fffce4705a9662cd70adb13d4a31832d,0,0,1,0,0,0,0


In [19]:
test['sum_status']=test.sum(axis=1)

In [20]:
test.loc[test['sum_status']!=1, :] 

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable,sum_status
order_id,,,,,,,,
002f98c0f7efd42638ed6100ca699b42,0,0,2,0,0,0,0,2
005d9a5423d47281ac463a968b3936fb,0,0,2,0,0,0,0,2
00946f674d880be1f188abc10ad7cf46,0,0,2,0,0,0,0,2
0097f0545a302aafa32782f1734ff71c,0,0,2,0,0,0,0,2
00bcee890eba57a9767c7b5ca12d3a1b,0,0,2,0,0,0,0,2
...,...,...,...,...,...,...,...,...
ffb18bf111fa70edf316eb0390427986,0,0,2,0,0,0,0,2
ffb8f7de8940249a3221252818937ecb,0,0,3,0,0,0,0,3
ffb9a9cd00c74c11c24aa30b3d78e03b,0,0,3,0,0,0,0,3


####  Auffälligkeiten
1. Die Nan Werte machen einen sehr geringen Anteil aus. 
2. Da die Zeilen in denen sich die NaN werte befinden, den Order Status delivered haben, werde ich die Daten berücksichten, da der Kauf stattgefunden.



Für die Auswertung von Aufgabe 1. ist der order_status sehr wichtig, da ich nur reale Bestellungen betrachten will.

Deswegen schau ich mir erstmal an wie sich die NaNs zu den relevanten Order Status verhalten

Für die RFM Analyse brauche ich nur die approved, delivered, invoiced, processing und shipped order_status
Daher kann ich canceled, created und unavailable erstmal rausnehmen, da dieser order_status für die Aufgabe nicht relevant ist.

In [21]:
valid_rfm_status = ['delivered', 'shipped', 'processing', 'invoiced', 'approved']
df_rfm_eda = df_rfm_eda[df_rfm_eda['order_status'].isin(valid_rfm_status)]

In [22]:
test.loc[test['sum_status']!=1, :].sort_values('sum_status', ascending=False)

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable,sum_status
order_id,,,,,,,,
ca3625898fbd48669d50701aba51cd5f,0,0,7,0,0,0,0,7
cf5c8d9f52807cb2d2f0a0ff54c478da,0,0,6,0,0,0,0,6
5a3b1c29a49756e75f1ef513383c0c12,0,0,6,0,0,0,0,6
b436eb981676e54c0bc9bcade0e079c4,0,0,5,0,0,0,0,5
bb82809ea3ca9f3edbe589b60e14e0cb,0,0,5,0,0,0,0,5
...,...,...,...,...,...,...,...,...
59b67c775c6a905fc4faac69ca74b5cb,0,0,2,0,0,0,0,2
59bccab4e9193a9229f7d1b73fcb47c3,0,0,2,0,0,0,0,2
59c0ed646a3b30d4054298988188486f,0,0,2,0,0,0,0,2


## Filterung nach der Bestellung mit den meisten Duplikaten

In [24]:
order_id = 'ca3625898fbd48669d50701aba51cd5f'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[df_rfm_eda['order_id'] == order_id]

order_data.head(63)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
28347,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,95.900002,0.15,2
29994,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,309.000000,1.84,1
58279,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,109.900002,0.15,1
65947,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,56.000000,3.68,2
78945,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,159.000000,3.67,2
85446,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,33.900002,1.84,1
87068,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,63.700001,0.15,1


In [26]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
count,101099,101099,101099.000000,101099.000000,101099.000000
mean,2018-01-01 04:49:34,2018-01-01 15:09:29,124.649025,20.140503,1.108824
min,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.000000
25%,2017-09-13 11:58:02,2017-09-13 20:10:21,40.799999,13.180000,1.000000
50%,2018-01-19 16:33:57,2018-01-20 09:08:37,79.000000,16.350000,1.000000
75%,2018-05-05 07:49:38,2018-05-05 14:13:51,139.000000,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.679993,20.000000
std,NaN,NaN,188.526642,15.891350,0.473023


### EDA für zweite Kernaufgabe

In [65]:
# Dataframe für Aufgabe 2. für EDA
df_pc_eda = sql("""
SELECT 
    p.product_id,
    pcnt.product_category_name_english,
    Count(o.order_id) AS order_count,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    oi.price,
    oi.freight_value,
    r.review_score
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id 
JOIN orders o ON oi.order_id = o.order_id
JOIN product_category_name_translation pcnt ON pcnt.product_category_name = p.product_category_name
LEFT JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status IN ('delivered', 'shipped', 'processing', 'invoiced', 'approved')
GROUP BY p.product_id, pcnt.product_category_name_english, o.order_status, o.order_purchase_timestamp,
         o.order_approved_at, oi.price, oi.freight_value, r.review_score
    """)

In [66]:
df_pc_eda

,product_id,product_category_name_english,order_count,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
0,4583e308182e4e78d5b71b5af1804def,baby,1,delivered,2017-09-19 20:03:29,2017-09-19 20:24:07,153.20,16.83,3
1,177d3d5bb9d4d29222a222e3b3554f41,stationery,1,delivered,2018-06-08 11:48:33,2018-06-08 12:21:35,118.50,21.25,5
2,4fe644d766c7566dbc46fb851363cb3b,art,1,delivered,2018-01-08 12:09:06,2018-01-08 12:17:25,139.99,25.92,1
3,c62dee961914cc2e49239963b04258ec,office_furniture,1,delivered,2017-02-09 22:05:38,2017-02-09 22:23:00,699.99,80.69,5
4,1cf22c0593f2ba8ecf404d0945a2036a,construction_tools_construction,1,delivered,2018-02-11 12:49:34,2018-02-15 03:51:36,128.80,34.26,4
...,...,...,...,...,...,...,...,...,...
100703,eebbed5ed3b134eceb717496c47652ba,bed_bath_table,1,delivered,2017-08-19 20:25:59,2017-08-22 04:05:17,99.99,48.23,<NA>
100704,16b691e994cb81b2e7e31c93ba603136,bed_bath_table,1,delivered,2018-05-11 18:52:04,2018-05-15 04:15:26,58.99,17.32,<NA>
100705,ba3a1e2c6cc1fb7a27dd74916212e6fb,toys,1,delivered,2018-01-24 23:12:41,2018-01-24 23:32:51,29.90,14.10,<NA>
100706,741a31499a578979be85db7f80139e62,furniture_decor,1,delivered,2017-12-14 00:06:50,2017-12-14 00:17:38,45.90,16.11,<NA>


In [67]:
df_pc_eda.describe()

,order_count,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
count,100708.000000,100708,100695,100708.000000,100708.000000,99940.0
mean,1.103597,2018-01-01 16:12:59.141210,2018-01-02 03:32:58.710333,124.151016,20.142170,4.088883
min,1.000000,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.0
25%,1.000000,2017-09-13 17:09:08,2017-09-14 02:45:40,40.140000,13.180000,4.0
50%,1.000000,2018-01-20 13:59:55.500000,2018-01-20 20:00:10,78.000000,16.360000,5.0
75%,1.000000,2018-05-05 21:22:12.750000,2018-05-06 13:50:11,139.000000,21.260000,5.0
max,20.000000,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.680000,5.0
std,0.461837,NaN,NaN,187.484910,15.898289,1.342309


In [68]:
df_pc_eda.dtypes

product_id                               object
product_category_name_english            object
order_count                               int64
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
price                                   float64
freight_value                           float64
review_score                              Int64
dtype: object

In [69]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'product_id': 'category',
              'product_category_name_english': 'category', 
              'order_count': 'Int16', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'price': 'float32',
              'freight_value': 'float32',
              'review_score': 'category',}
df_pc_eda = df_pc_eda.astype(col_dtypes)
df_pc_eda.dtypes

product_id                            category
product_category_name_english         category
order_count                              Int16
order_status                          category
order_purchase_timestamp         datetime64[s]
order_approved_at                datetime64[s]
price                                  float32
freight_value                          float32
review_score                          category
dtype: object

In [57]:
df_pc_eda.describe()

,order_purchase_timestamp,order_approved_at,payment_value,price,freight_value
count,116013,115999,116013.000000,116013.000000,116013.000000
mean,2017-12-31 04:53:31,2017-12-31 16:18:46,172.438751,120.449623,20.062504
min,2016-09-04 21:15:19,2016-10-04 09:43:32,0.000000,0.850000,0.000000
25%,2017-09-12 13:49:33,2017-09-12 21:02:44,61.000000,39.900002,13.080000
50%,2018-01-18 20:49:36,2018-01-19 10:55:40,108.120003,74.900002,16.320000
75%,2018-05-04 13:28:13,2018-05-04 19:35:16,189.559998,134.199997,21.219999
max,2018-09-03 09:06:57,2018-09-03 17:40:06,13664.080078,6735.000000,409.679993
std,NaN,NaN,266.087036,182.709961,15.834814


In [70]:
df_pc_eda.isna().sum()

product_id                         0
product_category_name_english      0
order_count                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 13
price                              0
freight_value                      0
review_score                     768
dtype: int64

In [71]:
df_pc_eda['order_approved_at'] = df_pc_eda['order_approved_at'].fillna(
    pd.to_datetime(df_pc_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_pc_eda[df_pc_eda.isna().any(axis=1)].head(15)

,product_id,product_category_name_english,order_count,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
1619,d493f5a208254fe85b3ab55d898461a1,food,6,delivered,2017-10-24 16:58:19,2017-10-24 17:14:28,20.490000,9.270000,NaN
1620,592962829d5a715304344e656e39108a,bed_bath_table,1,delivered,2017-11-28 16:27:19,2017-11-28 16:36:29,116.900002,13.160000,NaN
1621,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,1,shipped,2018-05-16 13:03:16,2018-05-16 13:19:59,149.899994,11.180000,NaN
1622,82b73150f90f4fef92913e35b9984bb7,housewares,1,delivered,2017-07-25 16:42:02,2017-07-25 16:55:09,109.989998,20.260000,NaN
1623,568b50dd27d5d76c97dc2f871cb93e9a,bed_bath_table,2,delivered,2017-05-21 11:02:17,2017-05-21 11:10:21,19.900000,11.850000,NaN
1624,f35927953ed82e19d06ad3aac2f06353,books_general_interest,1,delivered,2017-08-12 10:25:24,2017-08-12 13:23:17,115.000000,15.560000,NaN
1625,00be617b58175bf207fd35910d5097a4,bed_bath_table,1,delivered,2017-08-19 17:10:17,2017-08-20 17:15:21,48.900002,16.110001,NaN
1626,1dec4c88c685d5a07bf01dcb0f8bf9f8,auto,1,delivered,2018-03-13 23:17:04,2018-03-13 23:30:26,589.000000,54.189999,NaN
1627,0cf13ac73dbcf6586ba63b89dd1f780a,furniture_decor,1,delivered,2017-06-03 13:13:42,2017-06-03 13:25:12,47.000000,12.690000,NaN
1628,4a78f5e17416dcfec28c101279235838,health_beauty,1,shipped,2018-05-10 16:44:57,2018-05-10 17:09:29,99.900002,18.580000,NaN


In [72]:
print("Gesamte Duplikate:", df_pc_eda.duplicated().sum())

Gesamte Duplikate: 0


### EDA für dritte Kernaufgabe

In [75]:
# Dataframe für Aufgabe 2. für EDA
df_service_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_score,
    s.seller_id,
    s.seller_city,
    s.seller_state,
    c.customer_city,
    c.customer_state,              
    pcnt.product_category_name_english,
     
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
JOIN sellers s
        ON oi.seller_id = s.seller_id
JOIN customers c 
        ON c.customer_id = o.customer_id
JOIN products p 
        ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
    """)

In [76]:
df_service_eda

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english
0,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05,4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,franca,SP,office_furniture
1,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06,5,b8bc237ba3788b23da09c0f1f3a3288c,itajai,SC,sao bernardo do campo,SP,housewares
2,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13,5,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,sao paulo,SP,office_furniture
3,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10,5,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,mogi das cruzes,SP,office_furniture
4,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15,5,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,campinas,SP,home_confort
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110745,1a9543c90f188e2e4fb14327ad4a9c9b,delivered,2018-01-30 15:28:21,2018-01-31 15:30:30,2018-02-16 16:28:33,2018-03-16 20:03:53,2018-03-13,1,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,curitiba,PR,office_furniture
110746,808c7c69c2778bdf4689eee0286e2bef,canceled,2018-02-22 07:57:07,2018-02-22 08:10:27,NaT,NaT,2018-03-13,1,8a32e327fe2c1b3511609d81aaf9f042,sao paulo,SP,sao paulo,SP,furniture_decor
110747,3304c0c857a9c77a201a551f5a3cacb8,delivered,2017-05-04 12:57:21,2017-05-04 13:10:46,2017-05-05 11:50:12,2017-05-08 10:27:31,2017-05-25,5,ddd51ae8cda92f3995a51fc0f0f3eec7,rio de janeiro,RJ,rio de janeiro,RJ,housewares
110748,cb1f3a44e8b8527e16913306a4d3de2f,delivered,2018-08-07 09:03:02,2018-08-08 09:05:09,2018-08-08 15:01:00,2018-08-15 19:28:29,2018-08-24,4,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,porto alegre,RS,telephony


In [77]:
df_service_eda.describe()


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score
count,110750,110736,109605,108457,110750,110750.000000
mean,2018-01-01 12:44:37.691458,2018-01-02 00:18:39.936452,2018-01-05 14:22:58.117586,2018-01-15 00:21:00.601399,2018-01-25 08:59:40.301580,4.035395
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.000000
25%,2017-09-14 08:05:20,2017-09-14 13:30:22,2017-09-18 22:27:28,2017-09-26 23:18:38,2017-10-05 00:00:00,4.000000
50%,2018-01-20 22:39:33,2018-01-22 13:53:44,2018-01-24 23:32:35,2018-02-03 18:38:41,2018-02-16 00:00:00,5.000000
75%,2018-05-05 15:14:23.750000,2018-05-05 22:13:43,2018-05-08 15:07:00,2018-05-16 12:53:08,2018-05-28 00:00:00,5.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,5.000000
std,NaN,NaN,NaN,NaN,NaN,1.385325


In [78]:
df_service_eda.dtypes


order_id                                 object
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
review_score                              int64
seller_id                                object
seller_city                              object
seller_state                             object
customer_city                            object
customer_state                           object
product_category_name_english            object
dtype: object

In [80]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'order_id': 'category',
              'order_status': 'category',
              'order_approved_at': 'datetime64[s]',
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_delivered_carrier_date': 'datetime64[s]', 
              'order_delivered_customer_date': 'datetime64[s]',
              'order_estimated_delivery_date': 'datetime64[s]', 
              'review_score': 'category', 
              'seller_id': 'category',
              'seller_city': 'category',
              'seller_state': 'category',
              'customer_city': 'category',
              'customer_state': 'category', 
              'product_category_name_english': 'category',
              }
df_service_eda = df_service_eda.astype(col_dtypes)
df_service_eda.dtypes

order_id                              category
order_status                          category
order_purchase_timestamp         datetime64[s]
order_approved_at                datetime64[s]
order_delivered_carrier_date     datetime64[s]
order_delivered_customer_date    datetime64[s]
order_estimated_delivery_date    datetime64[s]
review_score                          category
seller_id                             category
seller_city                           category
seller_state                          category
customer_city                         category
customer_state                        category
product_category_name_english         category
dtype: object

In [81]:
df_service_eda.isna().sum()

order_id                            0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  14
order_delivered_carrier_date     1145
order_delivered_customer_date    2293
order_estimated_delivery_date       0
review_score                        0
seller_id                           0
seller_city                         0
seller_state                        0
customer_city                       0
customer_state                      0
product_category_name_english       0
dtype: int64

In [84]:
df_service_eda['order_approved_at'] = df_service_eda['order_approved_at'].fillna(
    pd.to_datetime(df_service_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_service_eda.isna().sum()

order_id                            0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   0
order_delivered_carrier_date     1145
order_delivered_customer_date    2293
order_estimated_delivery_date       0
review_score                        0
seller_id                           0
seller_city                         0
seller_state                        0
customer_city                       0
customer_state                      0
product_category_name_english       0
dtype: int64